# JATIKARYA – Server Aplikasi (Google Colab)
Jalankan sel dari atas ke bawah. Runtime → Change runtime type → **T4 GPU** (CPU juga bisa, lebih lambat).

## Step 1 · Hubungkan Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 · Ambil kode aplikasi dan pasang pustaka

In [ ]:
REPO = 'https://github.com/rahmathidyatt/jatikarya.git'
!rm -rf /content/jatikarya && git clone -q $REPO /content/jatikarya
%cd /content/jatikarya
!pip install -q -r requirements.txt

## Step 3 · Atur lokasi folder
`PHOTOS` = folder berisi **40 foto sumber asli** (sebelum diberi watermark), sama seperti yang dipakai di JATIKARYA v2.
`REGISTRY` = tempat menyimpan hasil, cukup dibuat sekali.

In [ ]:
PHOTOS   = '/content/drive/MyDrive/GANTI/folder_foto_sumber_asli'
REGISTRY = '/content/drive/MyDrive/JATIKARYA_APP/registry.npz'
import os; os.makedirs(os.path.dirname(REGISTRY), exist_ok=True)

## Step 4 · Bangun registry 40 foto sumber
Lewati sel ini kalau `registry.npz` sudah ada. Pastikan baris *Step 2* menampilkan **40/40**.

In [ ]:
import os
if not os.path.exists(REGISTRY):
    !python scripts/build_registry.py --photos "$PHOTOS" --out "$REGISTRY"
else:
    print('registry sudah ada:', REGISTRY)

## Step 5 · Nyalakan server

In [ ]:
import subprocess, time, requests, os
env = dict(os.environ, JATIKARYA_REGISTRY=REGISTRY)
server = subprocess.Popen(['uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'],
                          env=env, stdout=open('/content/server.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        h = requests.get('http://localhost:8000/api/health', timeout=2).json(); break
    except Exception:
        time.sleep(2)
print(h)
# ready False? lihat /content/server.log

## Step 6 · Buka akses dari browser (tanpa akun)
Salin alamat `https://….trycloudflare.com` yang muncul, lalu buka di HP atau laptop.

In [ ]:
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
import subprocess, re, time
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('BUKA APLIKASI DI:', m.group(0)); break

## Step 7 (opsional) · Uji satu gambar hasil edit dari penelitian
Bandingkan hasilnya dengan baris gambar yang sama di `naive_fusion_240_results.csv`.

In [ ]:
TEST = '/content/drive/MyDrive/GANTI/removal gpt 01.png'
r = requests.post('http://localhost:8000/api/verify', files={'file': open(TEST, 'rb')}).json()
print('Watermark :', r['stage3_trustmark']['status'])
print('Top-1     :', r['stage4_dinov2']['top'][0])
print('Skor      :', round(r['stage5_calibration']['score'], 4))
print('Keputusan :', r['stage6_decision'])

Server hidup selama sesi Colab aktif. Kalau sesi putus, jalankan lagi Step 1–2 lalu Step 5–6 (Step 4 tidak perlu diulang).